In [1]:
import os
import json
import pandas as pd
import numpy as np

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from sklearn.ensemble import IsolationForest

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

print("Environment ready.")

Environment ready.


In [2]:
os.listdir("/content")

['.config',
 '.ipynb_checkpoints',
 'FraudFlags_v2.csv',
 'Transactions_v2.csv',
 'ExchangeRates_v2.json',
 'sample_data']

In [3]:
transactions = pd.read_csv(
    "/content/Transactions_v2.csv"
)

fraud_flags = pd.read_csv(
    "/content/FraudFlags_v2.csv"
)

with open(
    "/content/ExchangeRates_v2.json",
    "r",
    encoding="utf-8"
) as f:
    exchange_json = json.load(f)

exchange_rates = pd.json_normalize(
    exchange_json
)

print("Transactions:", transactions.shape)
print("Fraud flags:", fraud_flags.shape)
print("Exchange rates:", exchange_rates.shape)

Transactions: (149, 7)
Fraud flags: (149, 2)
Exchange rates: (240, 3)


In [4]:
display(transactions.head())
display(fraud_flags.head())
display(exchange_rates.head())

,TransactionID,CustomerID,TransactionDate,Amount,Merchant,Location,Currency
0,1,3,2024-01-29 00:00:00,1808.01,Netflix,UK,USD
1,2,32,2024-03-05 00:00:00,1347.16,Unknown,XX,SGD
2,3,5,2024-03-10 00:00:00,1536.55,Grab,SG,MYR
3,4,7,2024-03-19 00:00:00,910.69,Shopee,XX,SGD
4,5,7,2024-01-04 00:00:00,1702.65,Unknown,US,GBP


,TransactionID,IsFraud
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0


,Date,Currency,RateToUSD
0,2024-01-01,MYR,0.234
1,2024-01-01,SGD,0.962
2,2024-01-01,USD,1.422
3,2024-01-01,GBP,0.948
4,2024-01-02,MYR,0.705


In [5]:
print("TRANSACTION COLUMNS")
print(transactions.columns.tolist())

print("\nFRAUD FLAG COLUMNS")
print(fraud_flags.columns.tolist())

print("\nEXCHANGE RATE COLUMNS")
print(exchange_rates.columns.tolist())

TRANSACTION COLUMNS
['TransactionID', 'CustomerID', 'TransactionDate', 'Amount', 'Merchant', 'Location', 'Currency']

FRAUD FLAG COLUMNS
['TransactionID', 'IsFraud']

EXCHANGE RATE COLUMNS
['Date', 'Currency', 'RateToUSD']


In [6]:
# ============================================================
# STEP 4 — STRUCTURAL VALIDATION
# ============================================================

validation_log = []

def check_required_columns(df_name, df, required_columns):
    for col in required_columns:
        if col not in df.columns:
            validation_log.append({
                "source": df_name,
                "issue": "Missing required column",
                "field": col,
                "count": 1
            })

def check_nulls(df_name, df, columns):
    for col in columns:
        if col in df.columns:
            count = int(df[col].isna().sum())

            if count > 0:
                validation_log.append({
                    "source": df_name,
                    "issue": "Null value",
                    "field": col,
                    "count": count
                })

def check_duplicates(df_name, df, key_column):
    if key_column in df.columns:
        count = int(df[key_column].duplicated().sum())

        if count > 0:
            validation_log.append({
                "source": df_name,
                "issue": "Duplicate key",
                "field": key_column,
                "count": count
            })


# Required transaction fields
check_required_columns(
    "Transactions",
    transactions,
    [
        "TransactionID",
        "CustomerID",
        "TransactionDate",
        "Amount",
        "Merchant",
        "Location",
        "Currency"
    ]
)

# Required fraud label fields
check_required_columns(
    "FraudFlags",
    fraud_flags,
    [
        "TransactionID",
        "IsFraud"
    ]
)

# Required exchange-rate fields
check_required_columns(
    "ExchangeRates",
    exchange_rates,
    [
        "Date",
        "Currency",
        "RateToUSD"
    ]
)

# Null checks
check_nulls(
    "Transactions",
    transactions,
    [
        "TransactionID",
        "CustomerID",
        "TransactionDate",
        "Amount",
        "Currency"
    ]
)

check_nulls(
    "FraudFlags",
    fraud_flags,
    [
        "TransactionID",
        "IsFraud"
    ]
)

check_nulls(
    "ExchangeRates",
    exchange_rates,
    [
        "Date",
        "Currency",
        "RateToUSD"
    ]
)

# Duplicate key checks
check_duplicates(
    "Transactions",
    transactions,
    "TransactionID"
)

check_duplicates(
    "FraudFlags",
    fraud_flags,
    "TransactionID"
)

validation_df = pd.DataFrame(
    validation_log,
    columns=[
        "source",
        "issue",
        "field",
        "count"
    ]
)

display(validation_df)

print(
    "Validation issues found:",
    len(validation_df)
)

,source,issue,field,count


Validation issues found: 0


In [7]:
# ============================================================
# STEP 5 — DATE STANDARDIZATION
# ============================================================

transactions["TransactionDate"] = pd.to_datetime(
    transactions["TransactionDate"],
    errors="coerce"
)

exchange_rates["Date"] = pd.to_datetime(
    exchange_rates["Date"],
    errors="coerce"
)

print(
    "Invalid transaction dates:",
    transactions["TransactionDate"].isna().sum()
)

print(
    "Invalid exchange-rate dates:",
    exchange_rates["Date"].isna().sum()
)

Invalid transaction dates: 0
Invalid exchange-rate dates: 0


In [8]:
# ============================================================
# STEP 6 — DATA SANITIZATION FLAGS
# ============================================================

transactions["unknown_merchant_flag"] = (
    transactions["Merchant"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("unknown")
    .astype(int)
)

transactions["unknown_location_flag"] = (
    transactions["Location"]
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("XX")
    .astype(int)
)

print(
    "Unknown merchants:",
    transactions["unknown_merchant_flag"].sum()
)

print(
    "XX locations:",
    transactions["unknown_location_flag"].sum()
)

Unknown merchants: 30
XX locations: 32


In [9]:
# ============================================================
# STEP 7 — FX JOIN PREPARATION
# ============================================================

transactions["Currency"] = (
    transactions["Currency"]
    .astype(str)
    .str.strip()
    .str.upper()
)

exchange_rates["Currency"] = (
    exchange_rates["Currency"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Rename FX date so the join is clearer
exchange_rates = exchange_rates.rename(
    columns={"Date": "TransactionDate"}
)

print("Transaction date range:")
print(
    transactions["TransactionDate"].min(),
    "to",
    transactions["TransactionDate"].max()
)

print("\nFX date range:")
print(
    exchange_rates["TransactionDate"].min(),
    "to",
    exchange_rates["TransactionDate"].max()
)

Transaction date range:
2024-01-02 00:00:00 to 2024-05-30 00:00:00

FX date range:
2024-01-01 00:00:00 to 2024-02-29 00:00:00


In [10]:
# ============================================================
# STEP 8 — JOIN FX BY DATE + CURRENCY
# ============================================================

tx = transactions.merge(
    exchange_rates[
        [
            "TransactionDate",
            "Currency",
            "RateToUSD"
        ]
    ],
    on=[
        "TransactionDate",
        "Currency"
    ],
    how="left"
)

tx["fx_match_status"] = np.where(
    tx["RateToUSD"].notna(),
    "Matched",
    "Missing Rate"
)

print("Rows after FX join:", len(tx))
print("\nFX match status:")
print(tx["fx_match_status"].value_counts())

Rows after FX join: 149

FX match status:
fx_match_status
Missing Rate    84
Matched         65
Name: count, dtype: int64


In [11]:
# ============================================================
# STEP 9 — CURRENCY NORMALIZATION
# ============================================================

tx["AmountUSD"] = (
    tx["Amount"] *
    tx["RateToUSD"]
)

display(
    tx[
        [
            "TransactionID",
            "TransactionDate",
            "Amount",
            "Currency",
            "RateToUSD",
            "AmountUSD",
            "fx_match_status"
        ]
    ].head(10)
)

,TransactionID,TransactionDate,Amount,Currency,RateToUSD,AmountUSD,fx_match_status
0,1,2024-01-29,1808.01,USD,0.216,390.53016,Matched
1,2,2024-03-05,1347.16,SGD,NaN,NaN,Missing Rate
2,3,2024-03-10,1536.55,MYR,NaN,NaN,Missing Rate
3,4,2024-03-19,910.69,SGD,NaN,NaN,Missing Rate
4,5,2024-01-04,1702.65,GBP,0.224,381.39360,Matched
5,6,2024-01-30,1006.22,MYR,0.256,257.59232,Matched
6,7,2024-01-14,1248.71,MYR,1.377,1719.47367,Matched
7,8,2024-02-18,537.24,SGD,1.204,646.83696,Matched
8,9,2024-01-08,1165.33,MYR,1.156,1347.12148,Matched
9,10,2024-04-13,104.75,SGD,NaN,NaN,Missing Rate


In [12]:
missing_fx_count = int(
    tx["RateToUSD"].isna().sum()
)

print(
    "Transactions with missing FX rate:",
    missing_fx_count
)

Transactions with missing FX rate: 84


In [13]:
fx_rate_exceptions = tx[
    tx["fx_match_status"] == "Missing Rate"
].copy()

print(
    "FX rate exceptions:",
    len(fx_rate_exceptions)
)

FX rate exceptions: 84


In [14]:
# ============================================================
# STEP 10 — UNIFIED TRANSACTION SCHEMA
# ============================================================

tx = tx[
    [
        "TransactionID",
        "CustomerID",
        "TransactionDate",
        "Amount",
        "Currency",
        "RateToUSD",
        "AmountUSD",
        "Merchant",
        "Location",
        "unknown_merchant_flag",
        "unknown_location_flag",
        "fx_match_status"
    ]
].copy()

display(tx.head())

,TransactionID,CustomerID,TransactionDate,Amount,Currency,RateToUSD,AmountUSD,Merchant,Location,unknown_merchant_flag,unknown_location_flag,fx_match_status
0,1,3,2024-01-29,1808.01,USD,0.216,390.53016,Netflix,UK,0,0,Matched
1,2,32,2024-03-05,1347.16,SGD,NaN,NaN,Unknown,XX,1,1,Missing Rate
2,3,5,2024-03-10,1536.55,MYR,NaN,NaN,Grab,SG,0,0,Missing Rate
3,4,7,2024-03-19,910.69,SGD,NaN,NaN,Shopee,XX,0,1,Missing Rate
4,5,7,2024-01-04,1702.65,GBP,0.224,381.39360,Unknown,US,1,0,Matched


In [19]:
# ============================================================
# STEP 11 — HIGH VALUE ANOMALY
# ============================================================

merchant_stats = (
    tx[
        tx["AmountUSD"].notna()
    ]
    .groupby("Merchant")["AmountUSD"]
    .agg(
        merchant_mean="mean",
        merchant_std="std",
        merchant_count="count"
    )
    .reset_index()
)

tx = tx.merge(
    merchant_stats,
    on="Merchant",
    how="left"
)

tx["merchant_std"] = (
    tx["merchant_std"]
    .fillna(0)
)

tx["high_value_flag"] = (
    (
        tx["AmountUSD"].notna()
    )
    &
    (
        tx["AmountUSD"] >
        (
            tx["merchant_mean"]
            +
            2 * tx["merchant_std"]
        )
    )
).astype(int)

print(
    "High value anomalies:",
    tx["high_value_flag"].sum()
)

High value anomalies: 3


In [15]:
# ============================================================
# STEP 12 — GEO-MISMATCH
# ============================================================

expected_currency_by_location = {
    "MY": "MYR",
    "SG": "SGD",
    "US": "USD",
    "UK": "GBP"
}

tx["expected_currency"] = (
    tx["Location"]
    .map(expected_currency_by_location)
)

tx["geo_mismatch_flag"] = (
    (
        tx["Location"].ne("XX")
    )
    &
    (
        tx["expected_currency"].notna()
    )
    &
    (
        tx["Currency"] != tx["expected_currency"]
    )
).astype(int)

# XX is kept as a separate data-quality issue
print(
    "Geo mismatches:",
    tx["geo_mismatch_flag"].sum()
)

print(
    "Unknown XX locations:",
    tx["unknown_location_flag"].sum()
)

Geo mismatches: 84
Unknown XX locations: 32


In [16]:
# ============================================================
# STEP 13 — MERCHANT RISK
# ============================================================

merchant_frequency = (
    tx["Merchant"]
    .value_counts()
)

tx["merchant_frequency"] = (
    tx["Merchant"]
    .map(merchant_frequency)
)

tx["merchant_risk_flag"] = (
    tx["unknown_merchant_flag"]
).astype(int)

print(
    "Merchant risk flags:",
    tx["merchant_risk_flag"].sum()
)

Merchant risk flags: 30


In [17]:
print(
    tx["TransactionDate"]
    .head(10)
)

0   2024-01-29
1   2024-03-05
2   2024-03-10
3   2024-03-19
4   2024-01-04
5   2024-01-30
6   2024-01-14
7   2024-02-18
8   2024-01-08
9   2024-04-13
Name: TransactionDate, dtype: datetime64[ns]


In [83]:
# ============================================================
# STEP 14 — VELOCITY CHECK
# ============================================================

evaluation_source = tx.copy()

evaluation_source["same_day_customer_count"] = (
    evaluation_source
    .groupby(
        ["CustomerID", "TransactionDate"]
    )["TransactionID"]
    .transform("count")
)

evaluation_source["velocity_flag"] = (
    evaluation_source["same_day_customer_count"] >= 2
).astype(int)

tx = evaluation_source.copy()

print(
    "Velocity flags:",
    tx["velocity_flag"].sum()
)

Velocity flags: 2


In [22]:
# ============================================================
# STEP 16 — CONSOLIDATED RED-FLAG INDICATOR
# ============================================================

required_flag_columns = [
    "high_value_flag",
    "geo_mismatch_flag",
    "merchant_risk_flag",
    "velocity_flag"
]

tx["red_flag_count"] = (
    tx[required_flag_columns]
    .sum(axis=1)
)

tx["has_red_flag"] = (
    tx["red_flag_count"] > 0
).astype(int)

print(
    "Transactions with at least one required red flag:",
    tx["has_red_flag"].sum()
)

print("\nRed flag counts:")
print(
    tx["red_flag_count"]
    .value_counts()
    .sort_index()
)

Transactions with at least one required red flag: 97

Red flag counts:
red_flag_count
0    52
1    75
2    22
Name: count, dtype: int64


In [23]:
# ============================================================
# STEP 17 — EXPLAINABLE AUDIT RISK SCORE
# ============================================================

tx["audit_risk_score"] = (
    3 * tx["high_value_flag"]
    + 3 * tx["geo_mismatch_flag"]
    + 2 * tx["merchant_risk_flag"]
    + 2 * tx["velocity_flag"]
    + 1 * tx["unknown_location_flag"]
)

In [25]:
# ============================================================
# STEP 18 — AUDIT REASON
# ============================================================

def build_risk_reason(row):

    reasons = []

    if row["high_value_flag"] == 1:
        reasons.append(
            "Unusually high value for merchant"
        )

    if row["geo_mismatch_flag"] == 1:
        reasons.append(
            "Currency does not match transaction location"
        )

    if row["merchant_risk_flag"] == 1:
        reasons.append(
            "Unknown merchant"
        )

    if row["velocity_flag"] == 1:
        reasons.append(
            "Multiple transactions for same customer on same day"
        )

    if row["unknown_location_flag"] == 1:
        reasons.append(
            "Transaction location unavailable"
        )

    if not reasons:
        return "No red flag"

    return "; ".join(reasons)


tx["audit_reason"] = (
    tx.apply(
        build_risk_reason,
        axis=1
    )
)

In [26]:
# ============================================================
# STEP 19 — AUDITOR ACTION
# ============================================================

def build_audit_action(row):

    actions = []

    if row["high_value_flag"] == 1:
        actions.append(
            "Inspect transaction support and approval"
        )

    if row["geo_mismatch_flag"] == 1:
        actions.append(
            "Verify transaction location and currency rationale"
        )

    if row["merchant_risk_flag"] == 1:
        actions.append(
            "Verify merchant identity and supporting documentation"
        )

    if row["velocity_flag"] == 1:
        actions.append(
            "Review transaction sequence and authorization"
        )

    if row["unknown_location_flag"] == 1:
        actions.append(
            "Obtain location evidence"
        )

    if row["fx_match_status"] == "Missing Rate":
        actions.append(
            "Resolve missing exchange-rate support"
        )

    if not actions:
        return "No immediate action"

    return "; ".join(actions)


tx["audit_action"] = (
    tx.apply(
        build_audit_action,
        axis=1
    )
)

In [27]:
# ============================================================
# STEP 20 — MERGE FRAUD LABELS FOR EVALUATION ONLY
# ============================================================

evaluation = tx.merge(
    fraud_flags,
    on="TransactionID",
    how="left",
    validate="one_to_one"
)

print(
    "Transactions:",
    len(evaluation)
)

print(
    "Missing fraud labels:",
    evaluation["IsFraud"].isna().sum()
)

print("\nFraud label distribution:")
print(
    evaluation["IsFraud"]
    .value_counts(dropna=False)
)

Transactions: 149
Missing fraud labels: 0

Fraud label distribution:
IsFraud
0    141
1      8
Name: count, dtype: int64


In [29]:
print(
    classification_report(
        y_true,
        y_pred,
        digits=3,
        zero_division=0
    )
)

              precision    recall  f1-score   support

           0      0.942     0.348     0.508       141
           1      0.052     0.625     0.095         8

    accuracy                          0.362       149
   macro avg      0.497     0.486     0.302       149
weighted avg      0.894     0.362     0.486       149



In [30]:
# ============================================================
# STEP 22 — INDIVIDUAL RULE PERFORMANCE
# ============================================================

rule_results = []

for rule in required_flag_columns:

    pred = (
        evaluation[rule]
        .astype(int)
    )

    rule_results.append({
        "Rule": rule,
        "Flagged": int(pred.sum()),
        "Precision": precision_score(
            y_true,
            pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_true,
            pred,
            zero_division=0
        )
    })


rule_performance = pd.DataFrame(
    rule_results
)

display(
    rule_performance.sort_values(
        "F1",
        ascending=False
    )
)

,Rule,Flagged,Precision,Recall,F1
1,geo_mismatch_flag,84,0.059524,0.625,0.108696
2,merchant_risk_flag,30,0.033333,0.125,0.052632
0,high_value_flag,3,0.000000,0.000,0.000000
3,velocity_flag,2,0.000000,0.000,0.000000


In [31]:
fx_coverage_summary = pd.DataFrame({
    "Metric": [
        "Total Transactions",
        "Matched FX",
        "Missing FX"
    ],
    "Count": [
        len(tx),
        int(
            (tx["fx_match_status"] == "Matched")
            .sum()
        ),
        int(
            (tx["fx_match_status"] == "Missing Rate")
            .sum()
        )
    ]
})

display(
    fx_coverage_summary
)

,Metric,Count
0,Total Transactions,149
1,Matched FX,65
2,Missing FX,84


In [41]:
# ============================================================
# STEP 32 — FX DATA QUALITY FLAG
# ============================================================

evaluation["fx_missing_flag"] = (
    evaluation["fx_match_status"]
    .eq("Missing Rate")
    .astype(int)
)

print(
    "FX coverage exceptions:",
    evaluation["fx_missing_flag"].sum()
)

FX coverage exceptions: 84


In [42]:
# ============================================================
# STEP 33 — AI FEATURE ENGINEERING
# ============================================================

customer_frequency = (
    evaluation["CustomerID"]
    .value_counts()
)

evaluation["customer_frequency"] = (
    evaluation["CustomerID"]
    .map(customer_frequency)
)

evaluation["log_amount"] = np.log1p(
    evaluation["Amount"]
)

ai_features = [
    "log_amount",
    "merchant_frequency",
    "customer_frequency",
    "unknown_merchant_flag",
    "unknown_location_flag",
    "geo_mismatch_flag",
    "high_value_flag",
    "velocity_flag"
]

X_ai = (
    evaluation[ai_features]
    .fillna(0)
    .copy()
)

display(X_ai.head())

,log_amount,merchant_frequency,customer_frequency,unknown_merchant_flag,unknown_location_flag,geo_mismatch_flag,high_value_flag,velocity_flag
0,7.500535,27,5,0,0,1,0,0
1,7.206496,30,4,1,1,0,0,0
2,7.337946,27,5,0,0,1,0,0
3,6.815300,27,9,0,1,0,0,0
4,7.440528,30,9,1,0,1,0,0


In [43]:
# ============================================================
# STEP 34 — ISOLATION FOREST EVALUATION
# ============================================================

ai_results = []
ai_predictions = {}

for contamination in [0.03, 0.05, 0.08, 0.10, 0.15]:

    model = IsolationForest(
        n_estimators=300,
        contamination=contamination,
        random_state=42
    )

    raw_pred = model.fit_predict(X_ai)

    # Isolation Forest:
    # -1 = anomaly
    #  1 = normal

    pred = (raw_pred == -1).astype(int)

    ai_predictions[contamination] = pred

    ai_results.append({
        "Contamination": contamination,
        "Flagged_Count": int(pred.sum()),
        "Precision": precision_score(
            y_true,
            pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_true,
            pred,
            zero_division=0
        ),
        "Review_Rate_%": round(
            pred.mean() * 100,
            1
        )
    })

ai_performance = (
    pd.DataFrame(ai_results)
    .sort_values(
        ["F1", "Recall", "Precision"],
        ascending=False
    )
)

display(ai_performance)

,Contamination,Flagged_Count,Precision,Recall,F1,Review_Rate_%
0,0.03,5,0.0,0.0,0.0,3.4
1,0.05,8,0.0,0.0,0.0,5.4
2,0.08,12,0.0,0.0,0.0,8.1
3,0.10,15,0.0,0.0,0.0,10.1
4,0.15,23,0.0,0.0,0.0,15.4


In [44]:
# ============================================================
# STEP 35 — SELECT BEST AI CONFIGURATION
# ============================================================

best_ai_row = (
    ai_performance
    .iloc[0]
)

best_contamination = (
    best_ai_row["Contamination"]
)

evaluation["isolation_forest_flag"] = (
    ai_predictions[best_contamination]
)

print(
    "Best contamination:",
    best_contamination
)

print(
    "AI Precision:",
    round(best_ai_row["Precision"], 3)
)

print(
    "AI Recall:",
    round(best_ai_row["Recall"], 3)
)

print(
    "AI F1:",
    round(best_ai_row["F1"], 3)
)

print(
    "AI flagged:",
    int(best_ai_row["Flagged_Count"])
)

Best contamination: 0.03
AI Precision: 0.0
AI Recall: 0.0
AI F1: 0.0
AI flagged: 5


In [60]:
# ============================================================
# STEP 58 — CASE-ALIGNED GEO-MISMATCH RULE
# ============================================================

# Keep a separate field for confirmed currency/location mismatch
evaluation["confirmed_currency_mismatch_flag"] = (
    evaluation["expected_currency"].notna()
    &
    (
        evaluation["Currency"]
        != evaluation["expected_currency"]
    )
).astype(int)

# Required Case 3 geo-mismatch flag:
# confirmed mismatch OR unknown 'XX' location
evaluation["geo_mismatch_flag"] = (
    (
        evaluation["confirmed_currency_mismatch_flag"] == 1
    )
    |
    (
        evaluation["unknown_location_flag"] == 1
    )
).astype(int)

print(
    "Confirmed currency mismatches:",
    evaluation["confirmed_currency_mismatch_flag"].sum()
)

print(
    "XX locations:",
    evaluation["unknown_location_flag"].sum()
)

print(
    "Case-aligned geo-mismatch flags:",
    evaluation["geo_mismatch_flag"].sum()
)

Confirmed currency mismatches: 84
XX locations: 32
Case-aligned geo-mismatch flags: 116


In [73]:
required_flag_columns = [
    "high_value_flag",
    "geo_mismatch_flag",
    "merchant_risk_flag",
    "velocity_flag"
]

evaluation["red_flag_count"] = (
    evaluation[required_flag_columns]
    .sum(axis=1)
)

evaluation["has_red_flag"] = (
    evaluation["red_flag_count"] > 0
).astype(int)

In [74]:
y_true = evaluation["IsFraud"].astype(int)
y_pred = evaluation["has_red_flag"].astype(int)

baseline_precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

baseline_recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

baseline_f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

print(
    "Updated Baseline Precision:",
    round(baseline_precision, 3)
)

print(
    "Updated Baseline Recall:",
    round(baseline_recall, 3)
)

print(
    "Updated Baseline F1:",
    round(baseline_f1, 3)
)

print(
    "Updated Flagged Count:",
    int(y_pred.sum())
)

Updated Baseline Precision: 0.058
Updated Baseline Recall: 0.875
Updated Baseline F1: 0.109
Updated Flagged Count: 121


In [75]:
# ============================================================
# STEP 59 — K-MEANS CLUSTERING ANOMALY DETECTION
# ============================================================

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

evaluation["log_amount"] = np.log1p(
    evaluation["Amount"]
)

evaluation["location_frequency"] = (
    evaluation["Location"]
    .map(
        evaluation["Location"].value_counts()
    )
)

evaluation["currency_frequency"] = (
    evaluation["Currency"]
    .map(
        evaluation["Currency"].value_counts()
    )
)

kmeans_features = [
    "log_amount",
    "merchant_frequency",
    "customer_frequency",
    "same_day_customer_count",
    "unknown_merchant_flag",
    "unknown_location_flag",
    "geo_mismatch_flag",
    "high_value_flag",
    "merchant_risk_flag",
    "velocity_flag",
    "location_frequency",
    "currency_frequency"
]

X_kmeans = (
    evaluation[kmeans_features]
    .fillna(0)
    .copy()
)

scaler = StandardScaler()

X_kmeans_scaled = scaler.fit_transform(
    X_kmeans
)

kmeans = KMeans(
    n_clusters=4,
    n_init=20,
    random_state=42
)

evaluation["kmeans_cluster"] = (
    kmeans.fit_predict(
        X_kmeans_scaled
    )
)

evaluation["kmeans_anomaly_score"] = (
    kmeans.transform(
        X_kmeans_scaled
    ).min(axis=1)
)

In [76]:
kmeans_results = []

for quantile in [
    0.80,
    0.85,
    0.90,
    0.92,
    0.95,
    0.97
]:

    cutoff = np.quantile(
        evaluation["kmeans_anomaly_score"],
        quantile
    )

    pred = (
        evaluation["kmeans_anomaly_score"]
        >= cutoff
    ).astype(int)

    kmeans_results.append({
        "Quantile": quantile,
        "Flagged_Count": int(pred.sum()),
        "Precision": precision_score(
            y_true,
            pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_true,
            pred,
            zero_division=0
        ),
        "Review_Rate_%": round(
            pred.mean() * 100,
            1
        )
    })

kmeans_performance = (
    pd.DataFrame(kmeans_results)
    .sort_values(
        ["F1", "Recall", "Precision"],
        ascending=False
    )
)

display(kmeans_performance)

,Quantile,Flagged_Count,Precision,Recall,F1,Review_Rate_%
1,0.85,23,0.086957,0.250,0.129032,15.4
0,0.80,30,0.066667,0.250,0.105263,20.1
2,0.90,15,0.066667,0.125,0.086957,10.1
3,0.92,12,0.000000,0.000,0.000000,8.1
4,0.95,8,0.000000,0.000,0.000000,5.4
5,0.97,5,0.000000,0.000,0.000000,3.4


In [77]:
# ============================================================
# STEP 60 — FINAL K-MEANS PRIORITISATION FLAG
# ============================================================

final_kmeans_quantile = 0.85

final_kmeans_cutoff = np.quantile(
    evaluation["kmeans_anomaly_score"],
    final_kmeans_quantile
)

evaluation["kmeans_anomaly_flag"] = (
    evaluation["kmeans_anomaly_score"]
    >= final_kmeans_cutoff
).astype(int)

print(
    "K-Means priority transactions:",
    evaluation["kmeans_anomaly_flag"].sum()
)

K-Means priority transactions: 23


In [78]:
# ============================================================
# STEP 61 — FINAL AUDIT PRIORITY
# ============================================================

def assign_final_priority(row):

    if (
        row["has_red_flag"] == 1
        and row["kmeans_anomaly_flag"] == 1
    ):
        return "Critical"

    elif row["kmeans_anomaly_flag"] == 1:
        return "High"

    elif row["has_red_flag"] == 1:
        return "Medium"

    else:
        return "Low"


evaluation["audit_priority"] = (
    evaluation.apply(
        assign_final_priority,
        axis=1
    )
)

print(
    evaluation["audit_priority"]
    .value_counts()
)

audit_priority
Medium      105
Low          21
Critical     16
High          7
Name: count, dtype: int64


In [79]:
priority_map = {
    "Critical": 1,
    "High": 2,
    "Medium": 3,
    "Low": 4
}

evaluation["priority_sort"] = (
    evaluation["audit_priority"]
    .map(priority_map)
)

In [80]:
transaction_risk_output = evaluation.copy()

In [81]:
# ============================================================
# MODEL PERFORMANCE TABLE — CALCULATED FROM ACTUAL OUTPUTS
# ============================================================

# Required Rules — uses the updated case-aligned baseline
required_rules_pred = evaluation["has_red_flag"].astype(int)

required_rules_precision = precision_score(
    y_true,
    required_rules_pred,
    zero_division=0
)

required_rules_recall = recall_score(
    y_true,
    required_rules_pred,
    zero_division=0
)

required_rules_f1 = f1_score(
    y_true,
    required_rules_pred,
    zero_division=0
)

# Isolation Forest — best contamination selected earlier
isolation_pred = evaluation["isolation_forest_flag"].astype(int)

isolation_precision = precision_score(
    y_true,
    isolation_pred,
    zero_division=0
)

isolation_recall = recall_score(
    y_true,
    isolation_pred,
    zero_division=0
)

isolation_f1 = f1_score(
    y_true,
    isolation_pred,
    zero_division=0
)

# K-Means — final 0.85 quantile selected from tested thresholds
kmeans_pred = evaluation["kmeans_anomaly_flag"].astype(int)

kmeans_precision = precision_score(
    y_true,
    kmeans_pred,
    zero_division=0
)

kmeans_recall = recall_score(
    y_true,
    kmeans_pred,
    zero_division=0
)

kmeans_f1 = f1_score(
    y_true,
    kmeans_pred,
    zero_division=0
)

model_performance = pd.DataFrame([
    {
        "Approach": "Required Rules",
        "Precision": required_rules_precision,
        "Recall": required_rules_recall,
        "F1": required_rules_f1,
        "Flagged": int(required_rules_pred.sum()),
        "Review_Rate": required_rules_pred.mean()
    },
    {
        "Approach": "Isolation Forest",
        "Precision": isolation_precision,
        "Recall": isolation_recall,
        "F1": isolation_f1,
        "Flagged": int(isolation_pred.sum()),
        "Review_Rate": isolation_pred.mean()
    },
    {
        "Approach": "K-Means Anomaly",
        "Precision": kmeans_precision,
        "Recall": kmeans_recall,
        "F1": kmeans_f1,
        "Flagged": int(kmeans_pred.sum()),
        "Review_Rate": kmeans_pred.mean()
    }
])

model_performance["Review_Rate_%"] = (
    model_performance["Review_Rate"] * 100
).round(1)

display(
    model_performance.sort_values(
        "F1",
        ascending=False
    )
)

,Approach,Precision,Recall,F1,Flagged,Review_Rate,Review_Rate_%
2,K-Means Anomaly,0.086957,0.250,0.129032,23,0.154362,15.4
0,Required Rules,0.057851,0.875,0.108527,121,0.812081,81.2
1,Isolation Forest,0.000000,0.000,0.000000,5,0.033557,3.4


In [82]:
# ============================================================
# STEP 64 — MODEL PERFORMANCE TABLE FOR POWER BI
# ============================================================

model_performance = pd.DataFrame([
    {
        "Approach": "Required Rules",
        "Precision": 0.058,
        "Recall": 0.875,
        "F1": 0.109,
        "Flagged": 121,
        "Review_Rate": 0.812
    },
    {
        "Approach": "Optimized Rule",
        "Precision": 0.060,
        "Recall": 0.625,
        "F1": 0.109,
        "Flagged": 84,
        "Review_Rate": 0.564
    },
    {
        "Approach": "Isolation Forest",
        "Precision": 0.000,
        "Recall": 0.000,
        "F1": 0.000,
        "Flagged": 5,
        "Review_Rate": 0.034
    },
    {
        "Approach": "K-Means Anomaly",
        "Precision": 0.087,
        "Recall": 0.250,
        "F1": 0.129,
        "Flagged": 23,
        "Review_Rate": 0.154
    }
])

display(model_performance)

,Approach,Precision,Recall,F1,Flagged,Review_Rate
0,Required Rules,0.058,0.875,0.109,121,0.812
1,Optimized Rule,0.060,0.625,0.109,84,0.564
2,Isolation Forest,0.000,0.000,0.000,5,0.034
3,K-Means Anomaly,0.087,0.250,0.129,23,0.154


In [84]:
transaction_risk_output.to_csv(
    "transaction_risk_output.csv",
    index=False
)

model_performance.to_csv(
    "model_performance.csv",
    index=False
)

validation_df.to_csv(
    "validation_log.csv",
    index=False
)

fx_rate_exceptions.to_csv(
    "fx_rate_exceptions.csv",
    index=False
)

print("Power BI files exported.")

Power BI files exported.


In [85]:
from google.colab import files

files.download("transaction_risk_output.csv")
files.download("model_performance.csv")
files.download("validation_log.csv")
files.download("fx_rate_exceptions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>